<a href="https://colab.research.google.com/github/sebenemaryamashebir-cmd/cassava-disease-detector/blob/Seben/Optimize.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Optimization Engineer

Improves on the baseline training run by experimenting with a small
grid of hyperparameters (learning rate, optimizer, weight decay, LR scheduler)
that don't require changing the CNN architecture itself. Each config trains for a few epochs; whichever
config gets the best VALIDATION F1 (macro-averaged, so the minority class
bacterial_blight isn't ignored) is then trained for the full epoch budget
and saved as the final "best" model, with a short markdown report describing
what was tried and why the winner won.

Run:
    python optimize.py --data-dir /path/to/cassava_dataset \
                        --search-epochs 3 --final-epochs 15
"""

import argparse
import copy
import json # to convert Python dictionaries into JSON text
import os
from datetime import datetime

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision import datasets, transforms
from sklearn.metrics import f1_score

from model import build_model

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

IMG_SIZE = 224
BATCH_SIZE = 32


In [ ]:
# Transforms
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.9, 1.0)),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
    transforms.ToTensor(),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

In [ ]:
# TransformSubset
class TransformSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        image, label = self.subset[idx]
        if self.transform is not None:
            image = self.transform(image)
        return image, label


In [ ]:
# build_dataloaders
def build_dataloaders(data_dir, batch_size=BATCH_SIZE):
    """Identical split logic to train.py/evaluate.py -- same seed, same
    ratios -- so the train/val split used here matches Member 2's exactly
    and the held-out test set stays untouched during this whole search."""
    base_dataset = datasets.ImageFolder(root=data_dir)
    class_names = [name.replace("Cassava___", "") for name in base_dataset.classes]
    num_classes = len(class_names)

    train_size = int(0.80 * len(base_dataset))
    val_size = int(0.10 * len(base_dataset))
    test_size = len(base_dataset) - train_size - val_size

    generator = torch.Generator().manual_seed(42)
    train_split, val_split, _test_split = random_split(
        base_dataset, [train_size, val_size, test_size], generator=generator
    )

    train_dataset = TransformSubset(train_split, train_transform)
    val_dataset = TransformSubset(val_split, eval_transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    # Class weights from the train split only, same rationale as train.py:
    # up-weight rare classes (bacterial_blight) in the loss so the model
    # can't just coast on always predicting the majority class (mosaic_disease).
    class_counts = torch.zeros(num_classes)
    for idx in train_split.indices:
        _, label = base_dataset.samples[idx]
        class_counts[label] += 1
    class_weights = 1.0 / class_counts.clamp(min=1)
    class_weights = class_weights / class_weights.sum() * num_classes

    return train_loader, val_loader, class_weights, class_names, num_classes


In [ ]:
# make_optimizer
def make_optimizer(name, params, lr, weight_decay):
    """Small factory so the search grid can swap optimizers by name."""
    if name == "adam": # adaptive learning rate
        return torch.optim.Adam(params, lr=lr, weight_decay=weight_decay)
    if name == "adamw":
        return torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    if name == "sgd":
        # momentum helps plain SGD converge at a comparable speed to Adam
        return torch.optim.SGD(params, lr=lr, momentum=0.9, weight_decay=weight_decay)
    raise ValueError(f"Unknown optimizer: {name}")


In [ ]:
# train_one_epoch and evaluate_epoch
def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total


@torch.no_grad()
def evaluate_epoch(model, loader, criterion, num_classes):
    """Same as train.py's evaluate(), but also returns macro-F1 so the
    search can rank configs by something that reflects minority-class
    performance, not just raw accuracy (which the imbalance can inflate)."""
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    macro_f1 = f1_score(all_labels, all_preds, average="macro",
                         labels=range(num_classes), zero_division=0)
    return running_loss / total, correct / total, macro_f1


In [ ]:
# run_search
def run_search(configs, train_loader, val_loader, class_weights, num_classes, search_epochs):
    """Train every candidate config for a short number of epochs and record
    its best validation macro-F1. This is a small grid search, not a full
    training run per config -- the point is to cheaply rank configs before
    committing the full epoch budget to just one."""
    results = []

    for cfg in configs:
        print(f"\n=== Trying config: {cfg} ===")
        torch.manual_seed(42)  # re-seed so every config starts from the same init weights
        model = build_model(num_classes=num_classes).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
        optimizer = make_optimizer(cfg["optimizer"], model.parameters(), cfg["lr"], cfg["weight_decay"])

        best_f1_this_cfg = 0.0
        for epoch in range(1, search_epochs + 1):
            tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
            va_loss, va_acc, va_f1 = evaluate_epoch(model, val_loader, criterion, num_classes)
            print(f"  epoch {epoch}: train acc {tr_acc:.3f} | val acc {va_acc:.3f} | val macro-F1 {va_f1:.3f}")
            best_f1_this_cfg = max(best_f1_this_cfg, va_f1)

        results.append({**cfg, "val_macro_f1": best_f1_this_cfg})

    results.sort(key=lambda r: r["val_macro_f1"], reverse=True)
    return results


In [ ]:
# train_final_model
def train_final_model(best_cfg, train_loader, val_loader, class_weights, num_classes, final_epochs, checkpoint_path):
    """Retrain the winning config for the full epoch budget, keeping the
    best-epoch checkpoint by validation macro-F1 (not just accuracy) since
    that's what the search selected on."""
    torch.manual_seed(42)
    model = build_model(num_classes=num_classes).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
    optimizer = make_optimizer(best_cfg["optimizer"], model.parameters(), best_cfg["lr"], best_cfg["weight_decay"])

    # Cosine-annealing LR schedule: smoothly decays the learning rate over
    # training so late epochs take smaller, more careful steps instead of
    # overshooting a good minimum. Cheap addition, commonly helps.
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=final_epochs)

    best_f1 = 0.0
    history = []
    for epoch in range(1, final_epochs + 1):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion)
        va_loss, va_acc, va_f1 = evaluate_epoch(model, val_loader, criterion, num_classes)
        scheduler.step()

        print(f"Epoch {epoch}/{final_epochs} | train acc {tr_acc:.3f} | "
              f"val acc {va_acc:.3f} | val macro-F1 {va_f1:.3f}")
        history.append({"epoch": epoch, "train_acc": tr_acc, "val_acc": va_acc, "val_macro_f1": va_f1})

        if va_f1 > best_f1:
            best_f1 = va_f1
            torch.save(model.state_dict(), checkpoint_path)
            print(f"  -> New best val macro-F1 ({va_f1:.3f}), saved to {checkpoint_path}")

    return best_f1, history


In [ ]:
# Main Program
# Colab equivalent of the argparse args in the original script
DATA_DIR = "/content/cassava_dataset"
SEARCH_EPOCHS = 3
FINAL_EPOCHS = 15
CHECKPOINT_DIR = "checkpoints"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

train_loader, val_loader, class_weights, class_names, num_classes = build_dataloaders(DATA_DIR)
print("Classes:", class_names)

In [ ]:
# search grid and run search
configs = [
    {"optimizer": "adam",  "lr": 1e-3, "weight_decay": 0.0},
    {"optimizer": "adam",  "lr": 1e-4, "weight_decay": 0.0},
    {"optimizer": "adamw", "lr": 1e-3, "weight_decay": 1e-4},
    {"optimizer": "sgd",   "lr": 1e-2, "weight_decay": 1e-4},
]

search_results = run_search(configs, train_loader, val_loader, class_weights, num_classes, SEARCH_EPOCHS)

print("\n=== Search results (best to worst, by val macro-F1) ===")
for r in search_results:
    print(r)

best_cfg = search_results[0]
print(f"\nBest config: {best_cfg}")

In [ ]:
# train final model and save report
final_checkpoint = os.path.join(CHECKPOINT_DIR, "best_model_optimized.pt")
best_f1, history = train_final_model(
    best_cfg, train_loader, val_loader, class_weights, num_classes,
    FINAL_EPOCHS, final_checkpoint,
)

report_path = os.path.join(CHECKPOINT_DIR, "optimization_report.md")
with open(report_path, "w") as f:
    f.write("# Optimization Report\n\n")
    f.write(f"Generated: {datetime.now().isoformat(timespec='seconds')}\n\n")
    f.write("## Search grid\n\n")
    f.write("| optimizer | lr | weight_decay | val_macro_f1 (search) |\n")
    f.write("|---|---|---|---|\n")
    for r in search_results:
        f.write(f"| {r['optimizer']} | {r['lr']} | {r['weight_decay']} | {r['val_macro_f1']:.4f} |\n")
    f.write(f"\n## Winning config\n\n`{json.dumps(best_cfg)}`\n\n")
    f.write(f"Trained for {FINAL_EPOCHS} epochs with a cosine-annealing LR schedule.\n")
    f.write(f"Best validation macro-F1 achieved: **{best_f1:.4f}**\n\n")
    f.write(f"Saved checkpoint: `{final_checkpoint}`\n\n")
    f.write("## Full training history\n\n")
    f.write("| epoch | train_acc | val_acc | val_macro_f1 |\n|---|---|---|---|\n")
    for h in history:
        f.write(f"| {h['epoch']} | {h['train_acc']:.4f} | {h['val_acc']:.4f} | {h['val_macro_f1']:.4f} |\n")

print(f"\nOptimization report saved to {report_path}")
print(f"Best model saved to {final_checkpoint}")